In [1]:
import os
import random
import warnings
from collections import defaultdict
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms as T
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2, FasterRCNN_ResNet50_FPN_V2_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')

# ── Configuration ──────────────────────────────────────────────────────────────
class CFG:
    ROOT = "/kaggle/input/competitions/amia-public-challenge-2026"
    TRAIN_DIR = f"{ROOT}/train/train"
    TEST_DIR = f"{ROOT}/test/test"
    TRAIN_CSV = f"{ROOT}/train.csv"
    IMG_SIZE_CSV = f"{ROOT}/img_size.csv"
    OUTPUT_DIR = "/kaggle/working"

    NUM_CLASSES = 15  # 14 + background
    TARGET_SIZE = 1024
    SEED = 40
    BATCH_SIZE = 4
    NUM_WORKERS = 2
    NUM_EPOCHS = 20
    LR = 0.005
    MOMENTUM = 0.9
    WEIGHT_DECAY = 0.0005
    LR_STEP_SIZE = 10
    LR_GAMMA = 0.1
    WBF_IOU_THRESH = 0.5
    OVERSAMPLE_FACTOR = 3
    RARE_CLASS_THRESHOLD = 500
    VAL_SPLIT = 0.2
    SCORE_THRESHOLD = 0.3
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    CLASS_NAMES = [
        "Aortic enlargement", "Atelectasis", "Calcification", "Cardiomegaly",
        "Consolidation", "ILD", "Infiltration", "Lung Opacity", "Nodule/Mass",
        "Other lesion", "Pleural effusion", "Pleural thickening", 
        "Pneumothorax", "Pulmonary fibrosis", "No finding"
    ]

def set_seed(seed=CFG.SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()

In [2]:
df = pd.read_csv(CFG.TRAIN_CSV)
img_size_df = pd.read_csv(CFG.IMG_SIZE_CSV)

print(f"Annotations: {len(df):,} | Images: {df['image_id'].nunique():,}")
print(f"Radiologists: {df['rad_id'].nunique()} | Classes: {df['class_name'].nunique()}")
print("\nClass distribution:")
print(df['class_name'].value_counts())

Annotations: 45,925 | Images: 8,573
Radiologists: 17 | Classes: 15

Class distribution:
class_name
No finding            15525
Aortic enlargement     5481
Pleural thickening     4308
Pulmonary fibrosis     4097
Cardiomegaly           4046
Nodule/Mass            2324
Pleural effusion       2190
Lung Opacity           2188
Other lesion           1945
Infiltration           1097
ILD                     904
Calcification           851
Consolidation           519
Atelectasis             255
Pneumothorax            195
Name: count, dtype: int64


In [3]:
class CoordinateScaler:
    def __init__(self, img_size_df, target_size=CFG.TARGET_SIZE):
        self.img_size = img_size_df.set_index('image_id').to_dict('index')
        self.target_size = target_size

    def scale_box(self, image_id, x_min, y_min, x_max, y_max):
        info = self.img_size.get(image_id, {'dim0': self.target_size, 'dim1': self.target_size})
        x_scale = self.target_size / info['dim1']
        y_scale = self.target_size / info['dim0']
        return x_min*x_scale, y_min*y_scale, x_max*x_scale, y_max*y_scale

scaler = CoordinateScaler(img_size_df)

In [5]:
def compute_iou(box1, box2):
    x1, y1 = max(box1[0], box2[0]), max(box1[1], box2[1])
    x2, y2 = min(box1[2], box2[2]), min(box1[3], box2[3])
    inter = max(0, x2-x1) * max(0, y2-y1)
    area1 = (box1[2]-box1[0])*(box1[3]-box1[1])
    area2 = (box2[2]-box2[0])*(box2[3]-box2[1])
    union = area1 + area2 - inter
    return inter/union if union > 0 else 0

def weighted_boxes_fusion(boxes_list, iou_thr=CFG.WBF_IOU_THRESH):
    if len(boxes_list) == 0: return [], []

    clusters = []
    used = [False] * len(boxes_list)

    for i in range(len(boxes_list)):
        if used[i]: continue
        cluster = [i]; used[i] = True
        for j in range(i+1, len(boxes_list)):
            if used[j]: continue
            for idx in cluster:
                if compute_iou(boxes_list[idx], boxes_list[j]) >= iou_thr:
                    cluster.append(j); used[j] = True; break
        clusters.append(cluster)

    fused = []
    for cluster in clusters:
        cboxes = [boxes_list[i] for i in cluster]
        n = len(cluster)
        avg = [
            sum(b[0] for b in cboxes)/n, sum(b[1] for b in cboxes)/n,
            sum(b[2] for b in cboxes)/n, sum(b[3] for b in cboxes)/n
        ]
        fused.append(avg)
    return fused, clusters

In [6]:
def build_consensus(df, scaler):
    rows = []
    for img_id, group in tqdm(df.groupby('image_id'), desc="WBF Consensus"):
        findings = group[group['class_id'] != 14]

        if len(findings) == 0:
            rows.append({'image_id': img_id, 'class_id': 14, 'class_name': 'No finding',
                        'x_min': None, 'y_min': None, 'x_max': None, 'y_max': None})
            continue

        for cid, cgroup in findings.groupby('class_id'):
            cname = cgroup['class_name'].iloc[0]
            boxes = []
            for _, r in cgroup.iterrows():
                if pd.isna(r['x_min']): continue
                boxes.append(scaler.scale_box(img_id, r['x_min'], r['y_min'], r['x_max'], r['y_max']))

            if not boxes: continue
            fused, _ = weighted_boxes_fusion(boxes)

            for box in fused:
                rows.append({'image_id': img_id, 'class_id': cid, 'class_name': cname,
                            'x_min': box[0], 'y_min': box[1], 'x_max': box[2], 'y_max': box[3]})

    return pd.DataFrame(rows)

consensus_df = build_consensus(df, scaler)
print(f"\nConsensus rows: {len(consensus_df):,}")
print(consensus_df['class_name'].value_counts())


WBF Consensus:   0%|          | 0/8573 [00:00<?, ?it/s]


Consensus rows: 25,647
class_name
No finding            5175
Pleural thickening    3586
Pulmonary fibrosis    2955
Aortic enlargement    2576
Cardiomegaly          1812
Lung Opacity          1782
Nodule/Mass           1694
Other lesion          1614
Pleural effusion      1570
Infiltration           838
ILD                    659
Calcification          658
Consolidation          409
Atelectasis            209
Pneumothorax           110
Name: count, dtype: int64


In [7]:
# Stratified split by primary class
img_classes = []
for img_id, g in consensus_df.groupby('image_id'):
    non_bg = g[g['class_id'] != 14]['class_id']
    primary = non_bg.mode().iloc[0] if len(non_bg) > 0 else 14
    img_classes.append((img_id, primary))

img_df = pd.DataFrame(img_classes, columns=['image_id', 'primary_class'])
train_ids, val_ids = train_test_split(
    img_df['image_id'].values, test_size=CFG.VAL_SPLIT, 
    random_state=CFG.SEED, stratify=img_df['primary_class'].values
)
print(f"Train: {len(train_ids):,} | Val: {len(val_ids):,}")

Train: 6,858 | Val: 1,715


In [8]:
def get_weights(consensus_df, image_ids):
    weights = []
    for img_id in image_ids:
        classes = consensus_df[(consensus_df['image_id']==img_id) & 
                               (consensus_df['class_id']!=14)]['class_id'].unique()
        if len(classes) == 0:
            weights.append(1.0)
        else:
            counts = consensus_df[consensus_df['class_id'].isin(classes)]['class_id'].value_counts()
            weights.append(CFG.OVERSAMPLE_FACTOR if counts.min() < CFG.RARE_CLASS_THRESHOLD else 1.0)
    return weights

train_weights = get_weights(consensus_df, train_ids)

In [9]:
import shutil

yolo_dir = f"{CFG.OUTPUT_DIR}/yolo_dataset"

def prep_yolo_split(image_ids, split):
    img_dir = os.path.join(yolo_dir, split, 'images')
    lbl_dir = os.path.join(yolo_dir, split, 'labels')
    os.makedirs(img_dir, exist_ok=True); os.makedirs(lbl_dir, exist_ok=True)

    for img_name in tqdm(image_ids, desc=f"YOLO {split}"):
        src = os.path.join(CFG.TRAIN_DIR, img_name + ".png")
        if os.path.exists(src): shutil.copy2(src, os.path.join(img_dir, img_name + ".png"))

        rows = consensus_df[consensus_df['image_id'] == img_name]
        lines = []
        for _, r in rows.iterrows():
            if r['class_id'] == 14: continue
            xc = ((r['x_min'] + r['x_max'])/2) / CFG.TARGET_SIZE
            yc = ((r['y_min'] + r['y_max'])/2) / CFG.TARGET_SIZE
            w = (r['x_max'] - r['x_min']) / CFG.TARGET_SIZE
            h = (r['y_max'] - r['y_min']) / CFG.TARGET_SIZE
            lines.append(f"{r['class_id']} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}")

        with open(os.path.join(lbl_dir, img_name + ".txt"), 'w') as f:
            f.write('\n'.join(lines))

prep_yolo_split(train_ids, 'train')
prep_yolo_split(val_ids, 'val')

# Create YAML
import yaml
yaml_path = os.path.join(yolo_dir, 'dataset.yaml')
with open(yaml_path, 'w') as f:
    yaml.dump({
        'path': yolo_dir, 'train': 'train/images', 'val': 'val/images',
        'names': {i: n for i, n in enumerate(CFG.CLASS_NAMES[:-1])}
    }, f, default_flow_style=False)


YOLO train:   0%|          | 0/6858 [00:00<?, ?it/s]

YOLO val:   0%|          | 0/1715 [00:00<?, ?it/s]

In [10]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 21.5 MB/s eta 0:00:00a 0:00:01


In [11]:
from ultralytics import YOLO

yolo_model = YOLO('yolov8m.pt')  # or yolov8x.pt for larger model

yolo_results = yolo_model.train(
    data=yaml_path,
    epochs=CFG.NUM_EPOCHS,
    imgsz=CFG.TARGET_SIZE,
    batch=CFG.BATCH_SIZE,
    device=0 if torch.cuda.is_available() else 'cpu',
    patience=5,
    save=True,
    project=f"{CFG.OUTPUT_DIR}/yolo_runs",
    name="chest_xray",
    exist_ok=True,
    mosaic=1.0,      # Enable mosaic augmentation for rare classes
    mixup=0.1,
    copy_paste=0.1,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=5, translate=0.1, scale=0.1, shear=2,
    fliplr=0.5,
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.67 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/yolo_dataset/dataset.yaml, degrees=5, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0

In [14]:
import os
import pandas as pd
import numpy as np
import torch
import cv2
from PIL import Image
from tqdm.auto import tqdm
from collections import defaultdict
from ultralytics import YOLO
import gc

# ── CONFIG ─────────────────────────────────────────────────────────────────────
YOLO_WEIGHTS = "/kaggle/working/yolo_runs/chest_xray/weights/best.pt"
TEST_DIR = "/kaggle/input/competitions/amia-public-challenge-2026/test/test"
OUTPUT_DIR = "/kaggle/working"
SCORE_THRESHOLD = 0.3
TARGET_SIZE = 1024
BATCH_SIZE= 4


def inference_yolo_native(weights_path, test_dir, output_csv="submission_yolo.csv"):
    """
    Use Ultralytics built-in predict(). Fastest and simplest.

    With imgsz=1024, boxes are returned in ABSOLUTE pixel coordinates
    in the 0-1024 range — directly usable for submission.
    """

    model = YOLO(weights_path)

    test_images = sorted([
        os.path.join(test_dir, f) 
        for f in os.listdir(test_dir) 
        if f.endswith('.png')
    ])

    results = []

    for start in range(0, len(test_images), BATCH_SIZE):

        batch = test_images[start:start+BATCH_SIZE]

        predictions = model.predict(
            source=batch,
            imgsz=TARGET_SIZE,      # Resizes to 1024x1024, boxes in 0-1024 range
            conf=0.01,              # Low threshold here, filter later
            save=False,
            stream= True, 
            verbose=False
        )
        for img_path, pred in zip(batch, predictions):
            img_name = os.path.basename(img_path)
            image_id = os.path.splitext(img_name)[0]
    
            pred_strings = []
    
            if pred.boxes is not None and len(pred.boxes) > 0:
                # boxes.xyxy: absolute pixel coords in resized image space [0, 1024]
                boxes = pred.boxes.xyxy.cpu().numpy()
                scores = pred.boxes.conf.cpu().numpy()
                labels = pred.boxes.cls.cpu().numpy().astype(int)
    
                for box, score, label in zip(boxes, scores, labels):
                    if float(score) < SCORE_THRESHOLD:
                        continue
    
                    xmin, ymin, xmax, ymax = box
                    # Clamp to valid range
                    xmin = max(0, xmin); ymin = max(0, ymin)
                    xmax = min(TARGET_SIZE, xmax); ymax = min(TARGET_SIZE, ymax)
    
                    pred_str = f"{label} {score:.4f} {xmin:.1f} {ymin:.1f} {xmax:.1f} {ymax:.1f}"
                    pred_strings.append(pred_str)
    
            if len(pred_strings) == 0:
                pred_strings = ["14 1.0 0 0 1 1"]
    
            results.append({
                "image_id": image_id,
                "PredictionString": " ".join(pred_strings)
            })

        del predictions
        gc.collect()
        
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        

    submission = pd.DataFrame(results)
    submission.to_csv(f"{OUTPUT_DIR}/{output_csv}", index=False)
    print(f"\\n✅ YOLO submission saved: {output_csv}")
    print(f"   Total images: {len(submission)}")
    print(submission.head())

    return submission

print("=" * 70)
print("YOLOv8 INFERENCE & SUBMISSION")
print("=" * 70)

print("\\n[1] Generating YOLO-only submission...")
yolo_sub = inference_yolo_native(
    weights_path=YOLO_WEIGHTS,
    test_dir=TEST_DIR,
    output_csv="submission_yolo.csv"
)


print("\\n" + "=" * 70)
print("DONE! Check /kaggle/working/ for submission CSV files.")
print("=" * 70)



YOLOv8 INFERENCE & SUBMISSION
\n[1] Generating YOLO-only submission...
\n✅ YOLO submission saved: submission_yolo.csv
   Total images: 6427
                           image_id                  PredictionString
0  00X4Pb5TcOhWWwrDwn9UoRDJhwYRuusp                    14 1.0 0 0 1 1
1  00eCz0yTwisqK7dgZKrdhLh4cMP9FewR                    14 1.0 0 0 1 1
2  00wsXaGGLhOo977BBHmhbKVNu02fWdPl                    14 1.0 0 0 1 1
3  02IEFam0BlSztSMY3YeA9svnDJOxTKDg  8 0.5870 705.1 355.0 861.7 538.2
4  02fQeJYiEhOeebwkwE8wsD0FPyz8EWHD                    14 1.0 0 0 1 1
\n======================================================================
DONE! Check /kaggle/working/ for submission CSV files.


In [3]:
import pandas as pd
submission_df = pd.read_csv("../submissions/02_submission.csv")
submission_df[submission_df['PredictionString'] == '14 1.0 0 0 1 1'].count()


image_id            5434
PredictionString    5434
dtype: int64